In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install pgmpy
!pip install pomegranate
!pip install pybbn

In [16]:
import pandas as pd # for data manipulation
import numpy as np
import networkx as nx # for drawing graphs
import matplotlib.pyplot as plt # for drawing graphs
from pgmpy.models import BayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination


# for creating Bayesian Belief Networks (BBN)
from pybbn.graph.dag import Bbn
from pybbn.graph.edge import Edge, EdgeType
from pybbn.graph.jointree import EvidenceBuilder
from pybbn.graph.node import BbnNode
from pybbn.graph.variable import Variable
from pybbn.pptc.inferencecontroller import InferenceController

#############################################################################
Monte Hall task

###################################################################################

In [17]:
import pandas as pd


# Read in the weather data csv
path = '/content/drive/MyDrive/2024_2025/PSM_2024_2025/Bayes_Fuzzy_Final/weatherAUS.csv'

data = pd.read_csv(path, delimiter=';')
# Drop records where target RainTomorrow=NaN
# Step 3: Select and prepare relevant columns
features = ['WindGustSpeed', 'Humidity9am', 'Humidity3pm', 'RainTomorrow']
df = data[features].dropna().copy()

# Calculate mean only for numeric columns and use it to fill missing values
#numeric_cols = df.select_dtypes(include=[np.number])  # This will filter only numeric columns
#means = numeric_cols.mean()
#df = df.fillna(means)

# Create bands for variables that we want to use in the model
df['WindGustSpeedCat'] = df['WindGustSpeed'].apply(lambda x: '0.<=40' if x <= 40 else
                                                             '1.40-50' if 40 < x <= 50 else '2.>50')
df['Humidity9amCat'] = df['Humidity9am'].apply(lambda x: '1.>60' if x > 60 else '0.<=60')
df['Humidity3pmCat'] = df['Humidity3pm'].apply(lambda x: '1.>60' if x > 60 else '0.<=60')

# Display the head of the dataframe to check results
print(df.head())


   WindGustSpeed  Humidity9am  Humidity3pm RainTomorrow WindGustSpeedCat  \
0           44.0         71.0         22.0           No          1.40-50   
1           44.0         44.0         25.0           No          1.40-50   
2           46.0         38.0         30.0           No          1.40-50   
3           24.0         45.0         16.0           No           0.<=40   
4           41.0         82.0         33.0           No          1.40-50   

  Humidity9amCat Humidity3pmCat  
0          1.>60         0.<=60  
1         0.<=60         0.<=60  
2         0.<=60         0.<=60  
3         0.<=60         0.<=60  
4          1.>60         0.<=60  


In [ ]:
# Create nodes by manually typing in probabilities
H9am = BbnNode(Variable(0, 'H9am', ['<=60', '>60']), [0.30658, 0.69342])
H3pm = BbnNode(Variable(1, 'H3pm', ['<=60', '>60']), [0.92827, 0.07173,
                                                      0.55760, 0.44240])
W = BbnNode(Variable(2, 'W', ['<=40', '40-50', '>50']), [0.58660, 0.24040, 0.17300])

parent_combinations = [
    ('<=60', '<=60', '<=40'),
    ('<=60', '<=60', '40-50'),
    ('<=60', '<=60', '>50'),
    ('<=60', '>60',  '<=40'),
    ('<=60', '>60',  '40-50'),
    ('<=60', '>60',  '>50'),
    ('>60',  '<=60', '<=40'),
    ('>60',  '<=60', '40-50'),
    ('>60',  '<=60', '>50'),
    ('>60',  '>60',  '<=40'),
    ('>60',  '>60',  '40-50'),
    ('>60',  '>60',  '>50')
]

# Probability for each combination
rt_probs = [
    (0.92314, 0.07686),
    (0.89072, 0.10928),
    (0.76008, 0.23992),
    (0.64250, 0.35750),
    (0.49168, 0.50832),
    (0.32182, 0.67818),
    (0.90100, 0.09900),
    (0.74300, 0.25700),
    (0.60000, 0.40000),
    (0.42000, 0.58000),
    (0.25000, 0.75000),
    (0.10000, 0.90000)
]

# Combine into flat list for BbnNode
flat_probs = [p for pair in rt_probs for p in pair]

# Create variablr and node RT
RT = BbnNode(
    Variable(3, 'RT', ['No', 'Yes']),
    flat_probs
)

In [ ]:
# This function helps to calculate probability distribution, which goes into BBN (note,
#can handle up to 2 parents)
def probs(data, child, parent1=None, parent2=None):
    if parent1==None:
        # Calculate probabilities
        prob=pd.crosstab(data[child], 'Empty', margins=False,
                         normalize='columns').sort_index().to_numpy().reshape(-1).tolist()
    elif parent1!=None:
            # Check if child node has 1 parent or 2 parents
            if parent2==None:
                # Caclucate probabilities
                prob=pd.crosstab(data[parent1],data[child], margins=False,
                                 normalize='index').sort_index().to_numpy().reshape(-1).tolist()
            else:
                # Caclucate probabilities
                prob=pd.crosstab([data[parent1],data[parent2]],data[child], margins=False,
                                 normalize='index').sort_index().to_numpy().reshape(-1).tolist()
    else: print("Error in Probability Frequency Calculations")
    return prob

In [ ]:
# Create nodes by using our earlier function to automatically calculate probabilities
H9am = BbnNode(Variable(0, 'H9am', ['<=60', '>60']), probs(df, child='Humidity9amCat'))
H3pm = BbnNode(Variable(1, 'H3pm', ['<=60', '>60']), probs(df, child='Humidity3pmCat',
                                                           parent1='Humidity9amCat'))
W = BbnNode(Variable(2, 'W', ['<=40', '40-50', '>50']), probs(df, child='WindGustSpeedCat'))
RT = BbnNode(Variable(3, 'RT', ['No', 'Yes']), probs(df, child='RainTomorrow',
                                                     parent1='Humidity3pmCat',
                                                     parent2='WindGustSpeedCat'))

# Create Network
bbn = Bbn() \
    .add_node(H9am) \
    .add_node(H3pm) \
    .add_node(W) \
    .add_node(RT) \
    .add_edge(Edge(H9am, H3pm, EdgeType.DIRECTED)) \
    .add_edge(Edge(H3pm, RT, EdgeType.DIRECTED)) \
    .add_edge(Edge(W, RT, EdgeType.DIRECTED))

# Convert the BBN to a join tree
join_tree = InferenceController.apply(bbn)

In [ ]:
# Set node positions
pos = {0: (-1, 2), 1: (-1, 0.5), 2: (1, 0.5), 3: (0, -1)}

# Set options for graph looks
options = {
    "font_size": 16,
    "node_size": 4000,
    "node_color": "white",
    "edgecolors": "black",
    "edge_color": "red",
    "linewidths": 5,
    "width": 5,}

# Generate graph
n, d = bbn.to_nx_graph()
nx.draw(n, with_labels=True, labels=d, pos=pos, **options)

# Update margins and print the graph
ax = plt.gca()
ax.margins(0.10)
plt.axis("off")
plt.show()

In [ ]:
# Define a function for printing marginal probabilities
def print_probs():
    for node in join_tree.get_bbn_nodes():
        potential = join_tree.get_bbn_potential(node)
        print("Node:", node)
        print("Values:")
        print(potential)
        print('----------------')

# Use the above function to print marginal probabilities
print_probs()

In [ ]:
# To add evidence of events that happened so probability distribution can be recalculated
def evidence(ev, nod, cat, val):
    ev = EvidenceBuilder() \
    .with_node(join_tree.get_bbn_node_by_name(nod)) \
    .with_evidence(cat, val) \
    .build()
    join_tree.set_observation(ev)

# Use above function to add evidence
evidence('ev1', 'H9am', '>60', 1.0)

# Print marginal probabilities
print_probs()

In [ ]:
# Add more evidence
evidence('ev1', 'H3pm', '>60', 1.0)
evidence('ev2', 'W', '>50', 1.0)
# Print marginal probabilities
print_probs()

###############################################################################
